In [62]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display
from loguru import logger

In [63]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins EEtSIsIa
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
Grok API Key not set (and this is optional)
Groq API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [64]:
# Connect to client libraries

google_client = OpenAI(base_url=os.getenv("GOOGLE_BASE_URL"), api_key=os.getenv("GOOGLE_API_KEY"))
openai_client = OpenAI()
ollama_client = OpenAI(api_key="ollama", base_url=os.getenv("OLLAMA_BASE_URL"))
hf_client = OpenAI(api_key=os.getenv("HF_API_KEY"), base_url=os.getenv("HF_BASE_URL"))

In [65]:
db_client = OpenAI(
    api_key=os.getenv("DATABRICKS_TOKEN"),
    base_url=os.getenv("DATABRICKS_BASE_URL")
)

In [ ]:
response = hf_client.chat.completions.create(
    model="deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    messages=[{"role": "user", "content": "Write hello world in Rust"}],
)

In [ ]:
print(response.choices[0].message.content )

In [ ]:
from llm_playground.rust_porting.system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
system_info

In [ ]:
rust_info = rust_toolchain_info()
rust_info

In [66]:
models = ["gpt-4.1-mini", "gpt-5.4", "openai/gpt-oss-20b:groq","gpt-5", "gemini-2.5-flash", "openai/gpt-oss-120b", "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "gemini-2.5-flash-lite", "moonshotai/Kimi-K2-Instruct:novita", "Qwen/Qwen3-Coder-Next:novita", "zai-org/GLM-5.1:together", "MiniMaxAI/MiniMax-M2.7:novita", "databricks-claude-sonnet-4-6", "databricks-claude-3-7-sonnet"]

clients = { "gpt-4.1-mini": openai_client, "gpt-5.4": openai_client, "gpt-5": openai_client, "gemini-2.5-flash": google_client, "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B": hf_client, "openai/gpt-oss-20b:groq": hf_client, "openai/gpt-oss-120b": hf_client, "gemini-2.5-flash-lite": google_client, "moonshotai/Kimi-K2-Instruct:novita": hf_client, "Qwen/Qwen3-Coder-Next:novita": hf_client, "zai-org/GLM-5.1:together": hf_client, "MiniMaxAI/MiniMax-M2.7:novita": hf_client, "databricks-claude-sonnet-4-6": db_client, "databricks-claude-3-7-sonnet": db_client }

# "claude-sonnet-4-5-20250929"
# "qwen2.5-coder", "deepseek-coder-v2", "qwen/qwen3-coder-30b-a3b-instruct"

In [ ]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openai_client.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

In [ ]:
compile_command = [
    "/Users/edan/.cargo/bin/rustc",
    "main.rs",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "codegen-units=1",
    "-C", "lto=fat",
    "-C", "panic=abort",
    "-C", "strip=symbols",
    "-o", "main.exe",
]

run_command = ["./main.exe"]

In [ ]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [ ]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [36]:
def port(model, python):
    try:
        client = clients[model]
        logger.info(f"Using model {model} with client {client}")
        reasoning_effort = "high" if 'gpt-5' in model else None
        if reasoning_effort:
            response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
        else:
            response = client.chat.completions.create(model=model, messages=messages_for(python))
        reply = response.choices[0].message.content
        reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
        return reply
    except Exception as e:
        raise gr.Error(f"Model error: {e}")

In [ ]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [ ]:
def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [ ]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

## Test running it

In [ ]:
code = port(models[1], python_hard)
print(code)

In [ ]:
run_python(python_hard)

In [ ]:
compile_and_run(code)

### An application for testing - Gradio

In [67]:
from llm_playground.rust_porting.styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True, show_error=True)


C:\Users\edan\AppData\Local\Temp\ipykernel_23340\3541690495.py:3: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


2026-04-21 01:48:34.586 | INFO     | __main__:port:4 - Using model databricks-claude-sonnet-4-6 with client <openai.OpenAI object at 0x00000157FE2B7050>
2026-04-21 01:48:52.600 | INFO     | __main__:port:4 - Using model openai/gpt-oss-20b:groq with client <openai.OpenAI object at 0x00000157FE2B5750>
2026-04-21 01:49:04.050 | INFO     | __main__:port:4 - Using model gpt-4.1-mini with client <openai.OpenAI object at 0x00000157FE2B5F50>
